In [ ]:
import numpy as np
from pathlib import Path
import skimage.measure as measure
import skimage.io as io
import skimage.morphology as morph
import skimage.segmentation as seg
import skimage.filters as filters
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
import plotly.express as px
from skimage.transform import resize, rescale
from scipy.ndimage import distance_transform_edt, gaussian_filter
from scipy.ndimage import uniform_filter1d, binary_closing, binary_opening
from tqdm.auto import tqdm
from scipy.spatial import Delaunay
import vedo
import pyclesperanto_prototype as pycle
import skimage.morphology as morph

In [ ]:
import numpy as np

def pad_3d_to_multiple(img, multiple, mode="constant", value=0):
    """
    Pad a 3D volume so z, y, x are multiples of `multiple`.

    Works for:
      img.shape = (z, y, x)
      img.shape = (z, y, x, c)  # with channels
    """
    z, y, x = img.shape[:3]

    pad_z = (-z) % multiple
    pad_y = (-y) % multiple
    pad_x = (-x) % multiple

    if img.ndim == 3:
        pads = (
            (0, pad_z),
            (0, pad_y),
            (0, pad_x),
        )
    elif img.ndim == 4:
        pads = (
            (0, pad_z),
            (0, pad_y),
            (0, pad_x),
            (0, 0),   # do not pad channels
        )
    else:
        raise ValueError("Expected img with shape (z,y,x) or (z,y,x,c)")

    if mode == "constant":
        return np.pad(img, pads, mode=mode, constant_values=value)

    return np.pad(img, pads, mode=mode)

In [ ]:
import einops

In [ ]:
folder = Path("/Users/schimmenti/Downloads/")
files = folder.glob("*.tif")

In [ ]:
img_filename = "/Users/schimmenti/Downloads/20241125_JF20_nubG4ecadGFP_UAS-SbNpRNAi_6hAPF_disc4_scale0.5_fused.tif"
image = io.imread(img_filename)
quantiles = np.quantile(image.flatten(), [0.01, 0.99])
image = np.clip(image, quantiles[0], quantiles[1])
image = (image - quantiles[0]) / (quantiles[1] - quantiles[0])
image = rescale(image, 0.5, anti_aliasing=True, preserve_range=True)

In [ ]:
image_filtered = filters.gaussian(image, sigma=1.0)
patch_size = 2
image_padded = pad_3d_to_multiple(image_filtered, multiple=patch_size, mode="constant", value=0)
#image_padded = np.ascontiguousarray(image_padded.swapaxes(2, 0).swapaxes(1, 0))  # (x, z, y)

In [ ]:
from einops import rearrange, einsum

p = patch_size
nx, ny, nz = np.array(image_padded.shape) // p

patches = rearrange(
    image_padded,
    "(nx xp) (ny yp) (nz zp) -> (nx ny nz) xp yp zp",
    xp=p, yp=p, zp=p,
)

# local voxel coordinates inside each patch
ii, jj, kk = np.meshgrid(
    np.arange(p),
    np.arange(p),
    np.arange(p),
    indexing="ij",
)

ijk = np.stack([ii, jj, kk], axis=-1).astype(float)  # (p, p, p, 3)

# patch origins in global coordinates
patch_ids = np.arange(patches.shape[0])
patch_indices = np.stack(np.unravel_index(patch_ids, (nx, ny, nz)), axis=-1)
patch_origins = patch_indices * p  # (n_patches, 3)
patch_centers = patch_origins + p / 2
# intensity / mass per patch
r = ijk - (p - 1) / 2   # centered coordinates inside patch

mass = patches.sum(axis=(1, 2, 3))
valid = mass > 0

dipole = np.full((patches.shape[0], 3), np.nan)
second = np.full((patches.shape[0], 3, 3), np.nan)
cov = np.full((patches.shape[0], 3, 3), np.nan)

dipole[valid] = einsum(
    patches[valid],
    r,
    "n x y z, x y z c -> n c",
) / mass[valid, None]

second[valid] = einsum(
    patches[valid],
    r,
    r,
    "n x y z, x y z c, x y z d -> n c d",
) / mass[valid, None, None]

cov[valid] = (
    second[valid]
    - dipole[valid, :, None] * dipole[valid, None, :]
)

vals = np.full((patches.shape[0],3), np.nan)
vecs = np.full((patches.shape[0],3,3), np.nan)
vals[valid], vecs[valid] = np.linalg.eigh(cov[valid])

In [ ]:
%matplotlib inline
from skimage.filters import threshold_otsu
for z_patches_index in range(nz):
    image_slice = image_padded[:,:,z_patches_index*p:(z_patches_index+1)*p].mean(axis=-1)
    subset_mask = (patch_indices[:, 2] == z_patches_index)*(valid)
    subset_centers = patch_centers[subset_mask]
    subset_dirs = dipole[subset_mask]
    subset_vals = vals[subset_mask]
    plt.figure(figsize=(10, 10))
    
    #qk = plt.quiver(
    #    subset_centers[:, 0],
    #    subset_centers[:, 1],
    #    subset_dirs[:, 0],
    #    subset_dirs[:, 1],
    #    np.log(subset_vals[:, -1]) - (1/3)*np.log(subset_vals.prod(axis=1)),
    #    angles="xy",
    #    scale_units="xy",
    #    scale=0.1/patch_size,
    #    cmap="coolwarm",
    #)
    scores = mass[subset_mask]*subset_vals.sum(axis=1)
    hist, edges = np.histogram(scores, bins=20)
    threshold = threshold_otsu(hist = (hist, edges[:-1] + np.diff(edges)/2))
    plt.scatter(subset_centers[:, 0], subset_centers[:, 1], c=scores > threshold, cmap="coolwarm", s=20)
    plt.imshow(image_slice.T, cmap="gray")
    plt.title(f"Patch covariances for z={z_patches_index}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.axis("equal")
    #plt.colorbar(label="Largest eigenvalue of covariance")
    plt.show()

In [ ]:
fig = go.Figure()
fig.update_layout(scene=dict(
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    zaxis=dict(visible=False),
    aspectmode='data'
))
fig.add_trace(go.Cone(
    x=patch_origins[valid, 0],
    y=patch_origins[valid, 1],
    z=patch_origins[valid, 2],
    u=(2*avg_ijk[valid, 0]-patch_size)/patch_size,
    v=(2*avg_ijk[valid, 1]-patch_size)/patch_size,
    w=(2*avg_ijk[valid, 2]-patch_size)/patch_size,
    sizemode="absolute",
    sizeref=10,
    opacity=1.0,
))
fig.show()

In [ ]:
for file in tqdm(list(files)):
    if not file.stem.startswith("20"):
        continue
    if not "ecadGFPnbG4_6hAPF_disc1" in file.stem:
        continue
    img = io.imread(file)
    labels = morph.label(img > 0)
    unique_labels, counts = np.unique(labels, return_counts=True)
    if unique_labels[0] == 0:
        unique_labels = unique_labels[1:]
        counts = counts[1:]
    largest_label = unique_labels[np.argmax(counts)]
    mask = labels == largest_label
    mask = binary_closing(mask, iterations=10)
    smoothed_mask = np.asarray(pycle.opening_labels(mask, radius=10))
    smoothed_mask = rescale(smoothed_mask, 0.5, preserve_range=True, anti_aliasing=True).astype(bool)
    vertices, faces, normals, values = measure.marching_cubes(smoothed_mask, level=0)
    vertices *= 2.0
    mesh = vedo.Mesh([vertices, faces])
    mesh = mesh.decimate_pro(fraction=0.9, preserve_topology=True)
    mesh = mesh.fill_holes()
    mesh = mesh.smooth(30)
    mesh.write(str(folder / f"{file.stem}_mesh.ply"), binary=False)

In [ ]:
vertices, triangles, vertex_normals, _ = measure.marching_cubes(smoothed_mask, level=0.0)
mesh = o3d.geometry.TriangleMesh()
mesh.vertices = o3d.utility.Vector3dVector(vertices)
mesh.triangles = o3d.utility.Vector3iVector(triangles)
mesh = o3d.geometry.TriangleMesh.filter_smooth_taubin(mesh, number_of_iterations=30)
mesh.compute_vertex_normals()
vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)
vertex_normals = np.asarray(mesh.vertex_normals)

In [ ]:
plt.imshow(erode_labels(mask, radius=1, iterations=20)[:,400,:])

In [ ]:
vertices, triangles, vertex_normals, _ = measure.marching_cubes(mask, level=0.0)
mesh = o3d.geometry.TriangleMesh()
mesh.vertices = o3d.utility.Vector3dVector(vertices)
mesh.triangles = o3d.utility.Vector3iVector(triangles)
mesh = o3d.geometry.TriangleMesh.filter_smooth_taubin(mesh, number_of_iterations=30)
mesh.compute_vertex_normals()
vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)
vertex_normals = np.asarray(mesh.vertex_normals)

In [ ]:
d_inside = distance_transform_edt(mask)
d_outside = distance_transform_edt(~mask)
sdf = d_outside - d_inside
sdf_smooth = gaussian_filter(sdf, sigma=1.0)

In [ ]:
from scipy.fft import dctn, idctn

def smooth_levelset_dct(phi0, lambdas, order=2):
    """
    Smooth scalar field by solving:

        min_phi 0.5 ||phi - phi0||^2
              + sum_a lambda_a / 2 ||D_a^order phi||^2

    lambdas are per axis, e.g. (z, y, x).
    Use large lambda on noisy axis.
    """
    shape = phi0.shape

    freqs = []
    for n in shape:
        k = np.arange(n)
        mu = 4.0 * np.sin(np.pi * k / (2.0 * n))**2
        freqs.append(mu)

    grids = np.meshgrid(*freqs, indexing="ij")

    denom = np.ones(shape)
    for lam, mu in zip(lambdas, grids):
        denom += lam * (mu ** order)

    phi_hat = dctn(phi0, type=2, norm="ortho")
    phi_smooth = idctn(phi_hat / denom, type=2, norm="ortho")

    return phi_smooth

In [ ]:
field = smooth_levelset_dct(sdf, lambdas=(1.0, 1.0, 50.0), order=2)

In [ ]:
plt.imshow(sdf[:,:,100] < 1)
plt.imshow(field[:,:,100] < 1, alpha=0.3, cmap="Reds")

In [ ]:
vertices, triangles, vertex_normals, _ = measure.marching_cubes(sdf_smooth < 1, level=0.0)
mesh = o3d.geometry.TriangleMesh()
mesh.vertices = o3d.utility.Vector3dVector(vertices)
mesh.triangles = o3d.utility.Vector3iVector(triangles)
mesh = o3d.geometry.TriangleMesh.simplify_quadric_decimation(mesh, target_number_of_triangles=500000)
mesh = o3d.geometry.TriangleMesh.filter_smooth_taubin(mesh, number_of_iterations=30)
mesh.compute_vertex_normals()
vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)
vertex_normals = np.asarray(mesh.vertex_normals)

In [ ]:
fig = go.Figure(data=[go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    colorscale='Viridis',
    flatshading=True,
    showscale=False,
    opacity=1.0,
)])
fig.update_layout(scene=dict(
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    zaxis=dict(visible=False),
    aspectmode='data'
))
fig.show()